# RQ1 — effort-reduction curve (TESSERA, 0%–100% expert)

**Purpose.** Reads and plots the full additive effort-reduction curve
(`docs/research_questions.md#rq1--citizen-science-as-a-substitute-for-expert-data`):
citizen-science presences as a fixed base, expert polygons added on top
in increasing amounts, to find where added field-collection effort
stops paying for itself. All 11 points (0%, 10%, …, 100%) are computed
under one fixed-test-set methodology, so the whole curve is
comparable, and every model is scored on the pooled test presences and
on each source separately. The two endpoints were first run on their
own in
[`presence_background_citizen_science.ipynb`](presence_background_citizen_science.ipynb)
and [`expert_arm_classifiers.ipynb`](expert_arm_classifiers.ipynb),
each with its own test-block split, so those numbers are not
comparable with the ones here.

**Where it is computed.** The models are trained and scored by
[`experiments/effort_reduction_curve/run.py`](../../experiments/effort_reduction_curve/run.py),
with fixed configuration and seeds, which writes the tables and a
`run.json` (configuration and hashes of the inputs) to
`results/effort_reduction_curve/`. This notebook only reads those
tables, draws the figure and holds the reading of the result. Run the
script again when the data or the parameters change.

**Data.** `inat_dealbata_clean.csv` (2021–2025 window, 2 465
presences), the expert polygons, and the uniform background pool
`background_points.npz`.

**Design decisions (mine, not yet confirmed by the supervisor):**

1. **Mixing unit: polygons, not pixels.** "X% expert" = a random X% of
   the expert *polygons* (every pixel inside them), not X% of pixels
   scattered across all polygons — collecting expert data costs a
   field visit per site/patch, not per pixel, so this is the effort
   unit that actually matches "cost of going to the field."
2. **Nested sampling.** Which polygons count as "effort already spent"
   is fixed by one random order (seed 0) for the whole curve, so 20%'s
   polygons are a superset of 10%'s — the curve reads as steadily
   adding effort, not redrawing a fresh random sample at every point.
3. **One fixed test set per evaluation seed, reused at every mixing
   level.** Test blocks are drawn once per seed (not once per level)
   from the 10 km blocks that contain citizen-science or expert
   presences, and held constant across all 11 levels within that seed
   — otherwise a score change across the curve could just mean
   "different test data," not "more expert data helped." Averaged over
   5 seeds.
4. **The test presence set always includes both sources**, even at
   100% (expert-only training), and is also scored by source.

**Classifiers:** Random Forest (main) and Logistic Regression (second
classifier), presence vs. background, same configuration as the
earlier notebooks (`n_estimators=300`, `class_weight="balanced"`;
`class_weight="balanced"`, `max_iter=1000`), no re-tuning. Metrics:
ROC-AUC and the continuous Boyce index.

> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> The text below the title was written under an earlier numbering:
> where it says "RQ2", read RQ1.

## 1. Load the saved run

The run is read from `results/effort_reduction_curve/`. The hashes of
the input files recorded in `run.json` are compared with the current
files: if one differs, the tables below were computed from other data
and the script has to be run again.

How the script builds the data: expert presence pixels are recovered
by a spatial join against `expert_polygons_clean.gpkg`, keeping for
every pixel **which polygon it came from**, needed for the
polygon-level mixing in Section 2. The one known overlapping polygon
pair (1.62 m², see `expert_polygons.ipynb`) can match a pixel to two
polygons; duplicates are dropped, keeping the first match, so every
pixel counts once.


In [ ]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src import manifest

OUT_DIR = ROOT / "results" / "effort_reduction_curve"
run = json.loads((OUT_DIR / "run.json").read_text())
source_df = pd.read_csv(OUT_DIR / "per_seed.csv")
source_summary = pd.read_csv(OUT_DIR / "summary.csv")
paired_df = pd.read_csv(OUT_DIR / "paired_differences.csv")
test_size_df = pd.read_csv(OUT_DIR / "test_sizes.csv")
reuse_df = pd.read_csv(OUT_DIR / "block_reuse.csv")
nominal_df = pd.read_csv(OUT_DIR / "nominal_levels.csv")
LEVELS = run["levels_pct"]

changed = [p for p, h in run["inputs_sha256"].items() if manifest.sha256_path(ROOT / p) != h]
if changed:
    warnings.warn(f"These inputs changed since the run of {run['run_date']}: {changed}. "
                  "Run experiments/effort_reduction_curve/run.py again.")
else:
    print(f"run of {run['run_date']} ({run['script']}): the input files are the current ones")

print(f"citizen-science presences: {run['n_cs_presences']:,} (matched {run['n_cs_matched']}/{run['n_cs_rows']})")
print(f"expert presence pixels: {run['n_expert_pixels']:,}, across {run['n_expert_polygons']:,} polygons")
print(f"background points: {run['n_background']:,}")

## 2. Effort order and mixing levels

A fixed random order over the expert polygons (seed 0, independent of
the 5 evaluation seeds in Section 3) decides which polygons "count" as
effort already spent at each level — this order is reused for every
evaluation seed, so the x-axis always means the same subset of
polygons, only the test/train split underneath it changes. Reported
`total_N` is nominal (full dataset, not one train fold) — the citizen-
science base (fixed) plus the expert pixels belonging to the included
polygons at that level, matching `docs/research_questions.md`'s
"report total N alongside the mixing percentage."


In [ ]:
nominal_df

## 3. Fixed spatial-block test set per seed

Test blocks are drawn once per seed from the blocks containing either
citizen-science or expert presences, and reused at every mixing level
within that seed (see "Design decisions" above). 20% of candidate
blocks held out, 5 seeds.


In [ ]:
print(f"{run['n_candidate_blocks']} candidate blocks (citizen-science ∪ expert presences)")
for seed, n in run["n_test_blocks_per_seed"].items():
    print(f"seed {seed}: {n} test blocks")

## 4. Scores by source of the test presences

For each seed and level, Random Forest and Logistic Regression are
trained once and scored three times against the same test background:
on all test presences (pooled), on the expert ones only and on the
citizen-science ones only. At 100% the citizen-science base is removed
and training is expert-only.

In [ ]:
print("Test presences per seed:")
display(test_size_df.round(3))
for metric in ["roc_auc", "boyce"]:
    print(f"{metric}, mean over 5 seeds, by level (% expert polygons):")
    display(source_summary.pivot_table(index=["classifier", "test_presences"], columns="level_pct",
                                       values=f"{metric}_mean", sort=False).round(3))

## 5. Reuse of expert test blocks across seeds

The seeds draw their test blocks independently, so the same blocks and pixels can enter the test set of more than one seed.

In [ ]:
# Are the seeds independent? Expert test blocks and pixels reused across seeds.
display(reuse_df.round(3))
print(f"{run['n_expert_blocks_in_a_test_set']} distinct blocks with expert polygons enter a test set, of {run['n_expert_blocks']}; "
      f"{run['n_expert_blocks_in_two_or_more_seeds']} of them in two seeds or more")
print(f"{run['n_expert_test_pixels_distinct']:,} distinct expert test pixels in {run['n_expert_test_pixels_summed_over_seeds']:,} summed over the seeds")

## 6. Paired differences per seed

Every level is compared with 0% inside each seed (same test blocks). The table for all levels is `paired_differences.csv` in `results/effort_reduction_curve/`.

In [ ]:
display(paired_df[paired_df["comparison"] == "90% minus 0%"].round(3))

## 7. Plot: ROC-AUC by level and test subset

In [ ]:
# Curve by test subset: small multiples with one shared y-axis. Colour = classifier.
import matplotlib.pyplot as plt

PLOT_COLORS = {"RandomForest": "#eb6834", "LogisticRegression": "#1baf7a"}
PLOT_MARKERS = {"RandomForest": "s", "LogisticRegression": "^"}
PANELS = [("pooled", "Pooled test presences"), ("expert only", "Expert presences only"),
          ("citizen science only", "Citizen-science presences only")]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (subset, title) in zip(axes, PANELS):
    for clf, color in PLOT_COLORS.items():
        sub = source_summary[(source_summary["classifier"] == clf) & (source_summary["test_presences"] == subset)].sort_values("level_pct")
        ax.plot(sub["level_pct"], sub["roc_auc_mean"], color=color, marker=PLOT_MARKERS[clf], markersize=6, linewidth=2, label=clf)
        ax.fill_between(sub["level_pct"], sub["roc_auc_mean"] - sub["roc_auc_sd"], sub["roc_auc_mean"] + sub["roc_auc_sd"],
                        color=color, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("% of expert polygons in training (100% = expert only)")
    ax.set_xticks(LEVELS)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)
axes[0].set_ylabel("ROC-AUC (mean ± sd over 5 seeds)")
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("Effort-reduction curve by test subset — TESSERA, 2021–2025 presences, current background pool", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "curve_by_source.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Reading the curve

- **Size of the expert test subset:** 103 to 589 expert pixels per
  seed (mean 424), 40% to 70% of the test presences in each seed
  (2 122 of 4 668, 45%, summed over the five seeds).
- **The independent units are polygons and 10 km blocks, not pixels,
  and seeds reuse blocks.** Each seed tests on 2 to 4 blocks with
  expert polygons. Over the five seeds only 9 of the 18 such blocks
  enter a test set, 6 of them in two seeds; the 2 122 expert test
  pixels are 1 265 distinct ones. In seeds 1 and 3 a single block
  gives 472 of the 523 and of the 557 expert test pixels. Agreement
  between seeds is therefore not independent evidence.
- **The 0% model does not score the same on the two sources.** Random
  Forest trained on citizen science only: ROC-AUC 0.976 on expert
  presences, 0.812 on citizen-science presences, 0.902 pooled (means;
  the expert value is the higher one in every seed).
- **The citizen-science-only model ranks the expert-mapped sites
  well** (0.976 for Random Forest, 0.927 for Logistic Regression).
  The higher value on expert presences may mean they are easier to
  separate from the background, not that the two sources are
  equivalent.
- **Random Forest, pooled:** 0.902 at 0%, then 0.906 to 0.908 at every
  level from 10% to 90%. The paired difference to 0% is about +0.005
  throughout; it is larger than its sd and positive in all five seeds
  at 20% to 50% (e.g. +0.005, sd 0.003, at 20%), and not at 60% to 90%
  (at 90%: +0.004, sd 0.005, four seeds up). A small step at the first
  10%, then flat.
- **Random Forest, expert presences only:** 0.976 → 0.986 (10%) →
  0.993 (90%); paired difference at 90% +0.017 (sd 0.005), all five
  seeds, and already +0.010 at 10%.
- **Random Forest, citizen-science presences only:** 0.812 → 0.801 at
  90% (−0.011, sd 0.013).
- **Logistic Regression gains more:** pooled 0.893 → 0.910 (10%) →
  0.924 (90%), +0.031 (sd 0.007); expert only 0.927 → 0.987, +0.060
  (sd 0.012); all five seeds in both.
- **The Boyce index moves the other way on expert presences** (Random
  Forest: 0.970 at 0%, 0.940 at 90%; lower in three seeds, equal in
  two; between 0.949 and 0.963 at the levels in between). With 10
  windows it takes few distinct values (0.969 in four seeds at 0%).
- **Expert-only training (100%)** keeps the expert subset high (Random
  Forest ROC-AUC 0.994, Boyce 0.961) and loses on citizen-science
  presences (0.677, sd 0.096; Boyce 0.585, sd 0.436). Pooled: 0.854.
  The drop at 100% in the curve comes from the citizen-science
  presences.

**Caveats:** ROC-AUC against background points is not an accuracy;
each subset is scored against the same test background, so the rows
of one model differ only in the presences.

**Background pool used.** `background_points.npz` as rebuilt on
2026-11-13 by `one_class_method_selection.ipynb`: same code, seed 42
and 2 000 points, with the 50 m exclusion buffer now drawn around the
2 465 presences of 2021–2025 (checked: the file is reproduced exactly
from that seed and the current presence file).